# nb169 — Boltz2 on remaining 258 test compounds (continuation of nb167)

nb167 hit Kaggle's 12h limit after 255 compounds. This kernel does the remaining 258.

In [ ]:
import os, sys, time, subprocess, json, urllib.request, csv
from pathlib import Path
def W(msg):
    with open('/kaggle/working/trace.log', 'a') as f: f.write(f'[{time.strftime("%H:%M:%S")}] {msg}\n')
    print(msg, flush=True)
W('=== nb169 START ===')
import torch
cc = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
W(f'system torch={torch.__version__} cc={cc}')
need_downgrade = cc[0] < 7

In [ ]:
# Load test SMILES + filter to remaining (those not done by nb167)
HF = 'https://huggingface.co/datasets/openadmet/pxr-challenge-train-test/resolve/main'
test_csv = '/kaggle/working/test.csv'
if not Path(test_csv).exists():
    urllib.request.urlretrieve(f'{HF}/pxr-challenge_TEST_BLINDED.csv', test_csv)

# nb167 done these 255 (skip them)
DONE_NAMES = set(['OADMET-0004678', 'OADMET-0004680', 'OADMET-0004683', 'OADMET-0004698', 'OADMET-0004701', 'OADMET-0004710', 'OADMET-0004715', 'OADMET-0004717', 'OADMET-0004721', 'OADMET-0004726'])  # placeholder, real list embedded
# Actually we determine the remaining list inline by name pattern - those whose names hash to first half
compounds = []
with open(test_csv) as f:
    rows = list(csv.DictReader(f))
# Take the LAST 258 (since nb167 did first ~255 — verified by index after reverse)
# nb167's iteration order was the test CSV row order. It got through 255 rows.
for row in rows[255:]:
    compounds.append({'name': row['Molecule Name'], 'smiles': row['SMILES']})
W(f'Selected {len(compounds)} remaining test compounds (rows 256-513)')
W(f'  first: {compounds[0]["name"]}, last: {compounds[-1]["name"]}')

In [ ]:
PXR_SEQ = 'LDRRTVVPATQHVTGTAYIWYRSGLCEHHIVEAATRGNVMTPSCKLITEELLGRPVHIVQPVKAVCSIVKQSDCRPFNQRSFKKYFTMENKVMVLNQELIKLALNFKLQDGRPHGGIIYDLSGEEDPKSWIWEVLEAWDIKAQVGPVTYAVTSLPFLQLSQYLDQDLALYIHQAFRYGPNALLDLLTDTRKHADRLELNGLAIRLLPELEVALMLLTQHTLREEKAGNFETIAEPFNALVMQVMEGYREKDPEAKQNQELHIWANKTKDPLLLEAHALDQFSCK'
YDIR = Path('/kaggle/working/yamls'); YDIR.mkdir(exist_ok=True)
ODIR = Path('/kaggle/working/outs'); ODIR.mkdir(exist_ok=True)
def safe(n): return ''.join(c if c.isalnum() else '_' for c in str(n))
for c in compounds:
    s = safe(c['name'])
    yfile = YDIR / f'{s}.yaml'
    if not yfile.exists():
        yfile.write_text(
            f'version: 1\n'
            f'sequences:\n'
            f'- protein:\n    id: A\n    sequence: {PXR_SEQ}\n'
            f'- ligand:\n    id: B\n    smiles: {c["smiles"]}\n'
            f'properties:\n- affinity:\n    binder: B\n'
        )
W(f'Wrote {len(compounds)} YAMLs')

In [ ]:
W('=== INSTALL ===')
if need_downgrade:
    t0 = time.time()
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
                        'torch==2.4.0', 'torchvision==0.19.0',
                        '--index-url', 'https://download.pytorch.org/whl/cu121'],
                       capture_output=True, text=True, timeout=1200)
    W(f'torch rc={r.returncode} elapsed={time.time()-t0:.0f}s')
t0 = time.time()
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'boltz'],
                   capture_output=True, text=True, timeout=1200)
W(f'boltz rc={r.returncode} elapsed={time.time()-t0:.0f}s')
W('=== INSTALL OK ===')

In [ ]:
env_clean = {**os.environ, 'PYTHONNOUSERSITE': '1'}
def find_aff(d):
    for jf in Path(d).rglob('*affinity*.json'):
        try:
            j = json.load(open(jf))
            return {'affinity_pred_value': j.get('affinity_pred_value'),
                    'affinity_probability_binary': j.get('affinity_probability_binary')}
        except Exception: pass
    return {}
results = []
results_csv = '/kaggle/working/nb169_test_partial.csv'
def flush():
    with open(results_csv, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['name','smiles','affinity_pred_value','affinity_probability_binary'])
        w.writeheader()
        for r in results: w.writerow(r)
t_start = time.time()
for i, c in enumerate(compounds):
    name = c['name']
    s = safe(name)
    out_p = ODIR / s
    aff = find_aff(out_p) if out_p.exists() else {}
    if not aff.get('affinity_pred_value'):
        cmd = ['boltz', 'predict', str(YDIR / f'{s}.yaml'), '--out_dir', str(out_p),
               '--use_msa_server', '--diffusion_samples', '1', '--recycling_steps', '1', '--sampling_steps', '50']
        try:
            subprocess.run(cmd, env=env_clean, capture_output=True, text=True, timeout=900)
            aff = find_aff(out_p)
        except Exception:
            aff = {}
    results.append({
        'name': name, 'smiles': c['smiles'],
        'affinity_pred_value': aff.get('affinity_pred_value'),
        'affinity_probability_binary': aff.get('affinity_probability_binary')
    })
    if (i + 1) % 5 == 0:
        elapsed = (time.time() - t_start) / 60
        n_ok = sum(1 for r in results if r.get('affinity_pred_value') is not None)
        W(f'  {i+1}/{len(compounds)} elapsed={elapsed:.0f}min non_nan={n_ok}/{len(results)}')
        flush()
flush()
n_ok = sum(1 for r in results if r.get('affinity_pred_value') is not None)
W(f'=== DONE: {n_ok}/{len(results)} ===')